In [1]:
#!/usr/bin/env python3
"""
STABLE FEATURE DISCOVERY + EWS DIAGNOSTICS (pre/post 1990)
=========================================================

Goal:
  Find (var, lon, mode, lag) features that explain AMOC well BOTH
  pre-1990 and post-1990, so they are meaningful/stable "dynamics"
  candidates for EWS diagnostics (rolling variance + lag-1 autocorr).

Assumptions:
  - EOF files are TRAIN-only basis, but contain PCs for FULL period by projection.
  - EOF netCDFs named: EOF_latdepth_{var}_{lon_tag}.nc and contain variable "PC".

Outputs:
  - stable_feature_ranking.csv
  - chosen_feature_timeseries.csv  (PC shifted by lag, aligned to AMOC years)
  - rolling_stats_feature_*.csv
  - plots: rolling variance and rolling lag-1 autocorr

Notes:
  - We work on detrended series by default (pre-fit detrend for both periods separately).
  - You can switch detrending off or change how detrend is applied below.
"""

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# ----------------------------
# Environment safety
# ----------------------------
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# =============================================================================
# USER CONFIG
# =============================================================================
MODEL  = "IPSL-CM6A-LR"   # "EC-Earth3" or "IPSL-CM6A-LR"
TARGET = "AMOC_45N_ensmean"
MODE   = "ensmean"          # "ensmean" or "member"
MEMBER_ID = None            # used only if MODE="member" (index or label)

EOF_DIR   = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/latdepth_sections/Train_period_85pct/"
AMOC_FILE = f"/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc"

OUTDIR = f"/data/users/frekle/AMOC_analysis/STABLE_FEATURES/{MODEL}/{TARGET}/{MODE}/"
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR:", OUTDIR)

# Candidate library settings
VARS     = ["thetao", "so"]
LON_TAGS = ["W10p0", "W20p0", "W30p0", "W40p0", "W60p0"]
N_MODES  = 10

MAX_LAG_ALLOWED = 20  # build candidates lags 0..MAX_LAG_ALLOWED

# Regime split
SPLIT_YEAR = 1990  # pre: <= 1989, post: >= 1990

# Preprocessing for correlation + EWS
DETREND_FOR_CORR = True
DETREND_MODE = "separate"  
# "separate": detrend within each regime separately (recommended for stability across regime)
# "pre_fit":  fit trend on pre regime, apply to all years (if you want a single baseline)
# "none":     no detrending for correlation

STANDARDIZE_FOR_CORR = True  # z-score within each regime before corr

# Feature filtering rules
REQUIRE_SAME_SIGN = True      # keep only features where corr_pre and corr_post have same sign
MIN_ABS_CORR_EACH = 0.10      # require |corr_pre| and |corr_post| >= threshold (after detrend/zscore)
MIN_POINTS_PRE  = 60
MIN_POINTS_POST = 20

# Selection for EWS plots
TOPK_STABLE = 10   # how many features to carry into rolling stats & plots

# Rolling EWS settings
ROLL_WINDOW_YEARS = 30  # typical 20-50; pick 30 as a start
ROLL_MIN_VALID_FRAC = 0.8  # require at least 80% finite in window

# Save figures
SAVE_FIGS = True
FIG_DPI = 250


# =============================================================================
# HELPERS
# =============================================================================
def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        t = np.asarray(time_coord)
        return np.array([int(str(x)[:4]) for x in t], dtype=int)

def load_pc(eof_dir, var, lon_tag, n_modes, mode="ensmean", member_id=None):
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon_tag}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")
    ds = xr.open_dataset(f)
    if "PC" not in ds:
        raise KeyError(f"'PC' not in {f}. Vars: {list(ds.data_vars)}")

    PC = ds["PC"].isel(mode=slice(0, int(n_modes)))

    if "member" in PC.dims:
        if mode == "ensmean":
            PC = PC.mean("member")
        elif mode == "member":
            if member_id is None:
                raise ValueError("member_id must be provided for mode='member'")
            if "member" in PC.coords and member_id in PC["member"].values:
                PC = PC.sel(member=member_id)
            else:
                PC = PC.isel(member=int(member_id))

    PC = PC.transpose("time", "mode")
    yrs = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", yrs)).swap_dims({"time": "year"}).drop_vars("time")
    ds.close()
    return PC.astype(float)

def load_amoc(amoc_file, target, mode="ensmean", member_id=None):
    ds = xr.open_dataset(amoc_file)
    if target not in ds.data_vars:
        raise KeyError(f"Target '{target}' not found. Available: {list(ds.data_vars)}")
    y = ds[target].squeeze()

    if "year" in y.dims:
        am = y
        if "time" in am.coords:
            am = am.drop_vars("time")
    else:
        yrs = extract_years(y["time"])
        am = y.assign_coords(year=("time", yrs)).swap_dims({"time": "year"}).drop_vars("time")

    if "member" in am.dims:
        if mode == "ensmean":
            am = am.mean("member")
        elif mode == "member":
            if member_id is None:
                raise ValueError("member_id must be provided for mode='member'")
            if "member" in am.coords and member_id in am["member"].values:
                am = am.sel(member=member_id)
            else:
                am = am.isel(member=int(member_id))

    ds.close()
    return am.astype(float).squeeze()

def corr_1d(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3:
        return np.nan
    aa = a[m] - np.mean(a[m])
    bb = b[m] - np.mean(b[m])
    denom = np.sqrt(np.sum(aa**2) * np.sum(bb**2))
    if denom == 0:
        return np.nan
    return float(np.sum(aa * bb) / denom)

def zscore(x):
    x = np.asarray(x, float)
    m = np.isfinite(x)
    if m.sum() < 3:
        return x.copy()
    mu = np.nanmean(x[m])
    sd = np.nanstd(x[m])
    if sd == 0 or not np.isfinite(sd):
        return x.copy()
    y = x.copy()
    y[m] = (y[m] - mu) / sd
    return y

def fit_trend(years, x):
    years = np.asarray(years, float)
    x = np.asarray(x, float)
    m = np.isfinite(years) & np.isfinite(x)
    if m.sum() < 3:
        return 0.0, float(np.nanmean(x))
    A = np.vstack([years[m], np.ones(m.sum())]).T
    a, b = np.linalg.lstsq(A, x[m], rcond=None)[0]
    return float(a), float(b)

def detrend_yearfit(years, x, fit_mask=None):
    years = np.asarray(years, float)
    x = np.asarray(x, float)
    if fit_mask is None:
        fit_mask = np.isfinite(years) & np.isfinite(x)
    a, b = fit_trend(years[fit_mask], x[fit_mask])
    trend = a * years + b
    return x - trend, (a, b)

def build_feature_series(pc_np, years, var, lon, mode0, lag):
    """
    Returns series aligned to years, same length as years, with NaNs for first 'lag' years.
    pc_np[(var,lon)] shape: (T, N_MODES) for aligned 'years'
    Feature value at time t is PC[t-lag, mode0].
    """
    T = len(years)
    out = np.full(T, np.nan, float)
    if lag < 0 or lag >= T:
        return out
    out[lag:] = pc_np[(var, lon)][0:T-lag, int(mode0)]
    return out

def rolling_var(x, w, min_valid):
    x = np.asarray(x, float)
    out = np.full_like(x, np.nan)
    for i in range(len(x)):
        j0 = max(0, i - w + 1)
        win = x[j0:i+1]
        m = np.isfinite(win)
        if m.sum() < min_valid:
            continue
        out[i] = float(np.nanvar(win[m], ddof=1))
    return out

def rolling_lag1_acf(x, w, min_valid):
    x = np.asarray(x, float)
    out = np.full_like(x, np.nan)
    for i in range(len(x)):
        j0 = max(0, i - w + 1)
        win = x[j0:i+1]
        m = np.isfinite(win)
        if m.sum() < min_valid:
            continue
        ww = win[m]
        if len(ww) < 3:
            continue
        a = ww[:-1]
        b = ww[1:]
        out[i] = corr_1d(a, b)
    return out


# =============================================================================
# LOAD + ALIGN
# =============================================================================
amoc = load_amoc(AMOC_FILE, TARGET, mode=MODE, member_id=MEMBER_ID)

pc_dict = {}
for var in VARS:
    for lon in LON_TAGS:
        pc_dict[(var, lon)] = load_pc(EOF_DIR, var, lon, N_MODES, mode=MODE, member_id=MEMBER_ID)

# intersect years
years = amoc["year"].values.astype(int)
for da in pc_dict.values():
    years = np.intersect1d(years, da["year"].values.astype(int))
years = np.asarray(years, int)
years.sort()

y = amoc.sel(year=years).values.astype(float)
pc_np = {k: v.sel(year=years).values.astype(float) for k, v in pc_dict.items()}

print(f"Aligned years: {years[0]}–{years[-1]} (T={len(years)})")

# masks
mask_pre  = years < SPLIT_YEAR
mask_post = years >= SPLIT_YEAR

if mask_pre.sum() < MIN_POINTS_PRE or mask_post.sum() < MIN_POINTS_POST:
    raise RuntimeError(
        f"Not enough points: pre={mask_pre.sum()} post={mask_post.sum()} "
        f"(need pre>={MIN_POINTS_PRE}, post>={MIN_POINTS_POST})"
    )


# =============================================================================
# PREPROCESS AMOC ONCE (for correlation)
# =============================================================================
y_for_corr = y.copy()

if DETREND_FOR_CORR and DETREND_MODE == "pre_fit":
    y_for_corr, _ = detrend_yearfit(years, y_for_corr, fit_mask=mask_pre)
elif DETREND_FOR_CORR and DETREND_MODE == "separate":
    y_pre_dt, _  = detrend_yearfit(years[mask_pre],  y_for_corr[mask_pre],  fit_mask=None)
    y_post_dt, _ = detrend_yearfit(years[mask_post], y_for_corr[mask_post], fit_mask=None)
    y_for_corr = y_for_corr.copy()
    y_for_corr[mask_pre]  = y_pre_dt
    y_for_corr[mask_post] = y_post_dt
elif DETREND_MODE == "none":
    pass

# standardize within each regime (optional)
if STANDARDIZE_FOR_CORR:
    y_for_corr = y_for_corr.copy()
    y_for_corr[mask_pre]  = zscore(y_for_corr[mask_pre])
    y_for_corr[mask_post] = zscore(y_for_corr[mask_post])


# =============================================================================
# BUILD + SCORE CANDIDATES
# =============================================================================
rows = []
lags = list(range(0, int(MAX_LAG_ALLOWED) + 1))

for var in VARS:
    for lon in LON_TAGS:
        X = pc_np[(var, lon)]  # (T, N_MODES)
        for mode0 in range(N_MODES):
            for lag in lags:
                feat = build_feature_series(pc_np, years, var, lon, mode0, lag)

                # detrend feature for correlation (match y treatment)
                f_for_corr = feat.copy()
                if DETREND_FOR_CORR and DETREND_MODE == "pre_fit":
                    f_for_corr, _ = detrend_yearfit(years, f_for_corr, fit_mask=mask_pre)
                elif DETREND_FOR_CORR and DETREND_MODE == "separate":
                    f_pre_dt, _  = detrend_yearfit(years[mask_pre],  f_for_corr[mask_pre],  fit_mask=None)
                    f_post_dt, _ = detrend_yearfit(years[mask_post], f_for_corr[mask_post], fit_mask=None)
                    f_for_corr = f_for_corr.copy()
                    f_for_corr[mask_pre]  = f_pre_dt
                    f_for_corr[mask_post] = f_post_dt

                if STANDARDIZE_FOR_CORR:
                    f_for_corr = f_for_corr.copy()
                    f_for_corr[mask_pre]  = zscore(f_for_corr[mask_pre])
                    f_for_corr[mask_post] = zscore(f_for_corr[mask_post])

                # compute correlations in each regime (ignore NaNs)
                corr_pre  = corr_1d(f_for_corr[mask_pre],  y_for_corr[mask_pre])
                corr_post = corr_1d(f_for_corr[mask_post], y_for_corr[mask_post])

                if not (np.isfinite(corr_pre) and np.isfinite(corr_post)):
                    continue

                # filters
                if abs(corr_pre) < MIN_ABS_CORR_EACH or abs(corr_post) < MIN_ABS_CORR_EACH:
                    continue
                if REQUIRE_SAME_SIGN and (np.sign(corr_pre) != np.sign(corr_post)):
                    continue

                stability = min(abs(corr_pre), abs(corr_post))
                mean_abs  = 0.5 * (abs(corr_pre) + abs(corr_post))
                gap_abs   = abs(abs(corr_pre) - abs(corr_post))  # lower = more consistent magnitude

                rows.append(dict(
                    var=var,
                    lon=lon,
                    mode=int(mode0 + 1),
                    mode0=int(mode0),
                    lag=int(lag),
                    corr_pre=float(corr_pre),
                    corr_post=float(corr_post),
                    stability=float(stability),
                    mean_abs=float(mean_abs),
                    gap_abs=float(gap_abs),
                ))

df = pd.DataFrame(rows)
if len(df) == 0:
    raise RuntimeError("No features survived filters. Try lowering MIN_ABS_CORR_EACH or turning off REQUIRE_SAME_SIGN.")

# rank: stability first, then mean_abs, then lowest gap_abs
df = df.sort_values(["stability", "mean_abs", "gap_abs"], ascending=[False, False, True]).reset_index(drop=True)

rank_csv = os.path.join(OUTDIR, "stable_feature_ranking.csv")
df.to_csv(rank_csv, index=False)
print("✅ Saved ranking:", rank_csv)

print("\nTop 10 stable features:")
print(df.head(10)[["var","lon","mode","lag","corr_pre","corr_post","stability","gap_abs"]])


# =============================================================================
# EWS DIAGNOSTICS FOR TOPK
# =============================================================================
top = df.head(int(TOPK_STABLE)).copy()

# Prepare AMOC (for plotting context only)
y_plot = y.copy()
# For EWS it’s usually better to remove long-term trend from the *indicator* series.
# We'll also provide detrended AMOC in the same way as DETREND_MODE for comparison.
if DETREND_FOR_CORR and DETREND_MODE == "pre_fit":
    y_plot, _ = detrend_yearfit(years, y_plot, fit_mask=mask_pre)
elif DETREND_FOR_CORR and DETREND_MODE == "separate":
    y_pre_dt, _  = detrend_yearfit(years[mask_pre],  y_plot[mask_pre],  fit_mask=None)
    y_post_dt, _ = detrend_yearfit(years[mask_post], y_plot[mask_post], fit_mask=None)
    y_plot = y_plot.copy()
    y_plot[mask_pre]  = y_pre_dt
    y_plot[mask_post] = y_post_dt

w = int(ROLL_WINDOW_YEARS)
min_valid = int(np.ceil(ROLL_MIN_VALID_FRAC * w))

all_series_rows = []

for i, r in top.iterrows():
    var = r["var"]; lon = r["lon"]; mode0 = int(r["mode0"]); lag = int(r["lag"])
    label = f"{var}_{lon}_EOF{mode0+1}_lag{lag}"

    feat = build_feature_series(pc_np, years, var, lon, mode0, lag)

    # detrend feature for EWS (generally yes)
    feat_dt = feat.copy()
    if DETREND_FOR_CORR and DETREND_MODE == "pre_fit":
        feat_dt, _ = detrend_yearfit(years, feat_dt, fit_mask=mask_pre)
    elif DETREND_FOR_CORR and DETREND_MODE == "separate":
        f_pre_dt, _  = detrend_yearfit(years[mask_pre],  feat_dt[mask_pre],  fit_mask=None)
        f_post_dt, _ = detrend_yearfit(years[mask_post], feat_dt[mask_post], fit_mask=None)
        feat_dt = feat_dt.copy()
        feat_dt[mask_pre]  = f_pre_dt
        feat_dt[mask_post] = f_post_dt

    # rolling stats on detrended feature
    rv = rolling_var(feat_dt, w=w, min_valid=min_valid)
    ra = rolling_lag1_acf(feat_dt, w=w, min_valid=min_valid)

    out_df = pd.DataFrame({
        "year": years.astype(int),
        "amoc": y_plot.astype(float),
        "feature": feat_dt.astype(float),
        "roll_var": rv.astype(float),
        "roll_acf1": ra.astype(float),
    })

    out_csv = os.path.join(OUTDIR, f"rolling_stats_{label}.csv")
    out_df.to_csv(out_csv, index=False)
    print("✅ Saved rolling stats:", out_csv)

    # store combined series rows (handy master table)
    all_series_rows.append(pd.DataFrame({
        "year": years.astype(int),
        "feature_name": label,
        "feature": feat_dt.astype(float),
    }))

    # plots
    if SAVE_FIGS:
        # Rolling variance
        plt.figure(figsize=(11, 4))
        plt.plot(years, rv)
        plt.axvline(SPLIT_YEAR, linestyle="--")
        plt.title(f"Rolling variance ({ROLL_WINDOW_YEARS}y) — {label}")
        plt.xlabel("Year")
        plt.ylabel("Var(feature)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        f1 = os.path.join(OUTDIR, f"FIG_rollvar_{label}.png")
        plt.savefig(f1, dpi=FIG_DPI, bbox_inches="tight")
        plt.close()

        # Rolling lag-1 autocorr
        plt.figure(figsize=(11, 4))
        plt.plot(years, ra)
        plt.axvline(SPLIT_YEAR, linestyle="--")
        plt.title(f"Rolling lag-1 autocorr ({ROLL_WINDOW_YEARS}y) — {label}")
        plt.xlabel("Year")
        plt.ylabel("ACF(1)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        f2 = os.path.join(OUTDIR, f"FIG_rollacf1_{label}.png")
        plt.savefig(f2, dpi=FIG_DPI, bbox_inches="tight")
        plt.close()

# Save a master “feature time series” table (long format)
all_series = pd.concat(all_series_rows, ignore_index=True)
series_csv = os.path.join(OUTDIR, f"chosen_feature_timeseries_top{TOPK_STABLE}.csv")
all_series.to_csv(series_csv, index=False)
print("✅ Saved feature timeseries:", series_csv)

print("\nDone.")


OUTDIR: /data/users/frekle/AMOC_analysis/STABLE_FEATURES/IPSL-CM6A-LR/AMOC_45N_ensmean/ensmean/


/dmidata/users/frekle/miniforge3/envs/aimoc_env/lib/python3.11/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'gini' loading failed:
cannot import name 'cartopy_utils' from partially initialized module 'metpy.plots' (most likely due to a circular import) (/dmidata/users/frekle/miniforge3/envs/aimoc_env/lib/python3.11/site-packages/metpy/plots/__init__.py)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)


Aligned years: 1850–2014 (T=165)
✅ Saved ranking: /data/users/frekle/AMOC_analysis/STABLE_FEATURES/IPSL-CM6A-LR/AMOC_45N_ensmean/ensmean/stable_feature_ranking.csv

Top 10 stable features:
      var    lon  mode  lag  corr_pre  corr_post  stability   gap_abs
0  thetao  W40p0     3    0  0.536968   0.532452   0.532452  0.004515
1      so  W30p0     1    4 -0.529103  -0.536174   0.529103  0.007070
2      so  W30p0     1    3 -0.523834  -0.546179   0.523834  0.022344
3  thetao  W20p0     9    4 -0.519228  -0.556877   0.519228  0.037650
4  thetao  W30p0     7   10  0.503583   0.520529   0.503583  0.016946
5  thetao  W20p0     9    5 -0.491353  -0.517693   0.491353  0.026340
6  thetao  W10p0     2   11 -0.478466  -0.487455   0.478466  0.008989
7  thetao  W10p0     2   10 -0.476236  -0.477996   0.476236  0.001760
8  thetao  W30p0     7    9  0.463347   0.578670   0.463347  0.115322
9      so  W40p0    10   20 -0.444211  -0.455478   0.444211  0.011267
✅ Saved rolling stats: /data/users/frekle